# Laboratorio 7: Proceso ETL - Mini Data WareHouse - Graficos

Bienvenido al Laboratorio 7. En esta sesión construiremos un pipeline de datos que extrae información real de **IMDb** mediante Web Scraping y la integra con datos sintéticos generados con **Faker**. Aprenderás a gestionar el ciclo de vida del dato desde su estado crudo hasta su carga en un Data Warehouse.

## 📋 Tabla de Contenidos

1. [Introducción al Proceso ETL](#-introduccion)
2. [Configuración de Infraestructura de Carpetas](#-setup)
3. [Fuente 1: Web Scraping IMDb (Extracción y Limpieza)](#-fuente1)
4. [Fuente 2: Generación de Visitantes con Faker](#-fuente2)
5. [Integración y Transformación Final](#-union)
6. [Carga en Mini Data Warehouse (SQLite3)](#-warehouse)
7. [Visualizaciones: BI desde SQL](#-viz)

---

<a id='-introduccion'></a>
## 🔹 1: Introducción al Proceso ETL

El proceso **ETL** (Extract, Transform, Load) es el flujo estándar en Ingeniería de Datos:
- **Extract:** Captura de datos desde fuentes externas (Web, CSV, DBs).
- **Transform:** Limpieza de nulos, corrección de tipos y normalización.
- **Load:** Persistencia en un sistema optimizado para consulta (Data Warehouse).

---

<a id='-setup'></a>
## 🔹 2: Configuración de Infraestructura

Organizaremos nuestro flujo de trabajo creando carpetas específicas para cada estado del dato.

In [ ]:
import os

# Crear estructura de carpetas
for folder in ['RawData', 'CleanData', 'TransformData']:
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f"Directorio listo: {folder}")

---

<a id='-fuente1'></a>
## 🔹 3: Fuente 1: Web Scraping IMDb

En esta etapa extraemos la lista de las mejores películas directamente de la web de IMDb.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. EXTRACCIÓN
url = 'https://www.imdb.com/list/ls024149810/'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
}

response = requests.get(url, headers=headers)
with open('RawData/peliculas_imdb.html', 'w', encoding='utf-8') as f:
    f.write(response.text)

# 2. TRANSFORMACIÓN (Parsing)
soup = BeautifulSoup(response.text, 'html.parser')
li_peliculas = soup.find_all('li', class_='ipc-metadata-list-summary-item')

titulos, ratings, years = [], [], []
for li in li_peliculas:
    titulo = li.find('h3', class_='ipc-title__text').text.strip()
    rating = li.find('span', class_='ipc-rating-star--rating').text.strip()
    metadata = li.find_all('span', class_='dli-title-metadata-item')
    year = metadata[0].text.strip() if metadata else "N/A"
    
    titulos.append(titulo)
    ratings.append(rating)
    years.append(year)

df_movies = pd.DataFrame({'Título': titulos, 'Rating_IMDb': ratings, 'Año': years})

# 3. GUARDADO (Clean Data)
df_movies.to_csv('CleanData/peliculas_imdb_clean.csv', index=False)
print(f"✅ IMDb: {len(df_movies)} películas extraídas.")

---

<a id='-fuente2'></a>
## 🔹 4: Fuente 2: Generación con Faker

Generamos 1000 registros de visitantes ficticios para simular la asistencia a un cine.

In [ ]:
from faker import Faker
import random

fake = Faker()
movies_list = df_movies['Título'].tolist()

def generar_registro():
    genero = random.choice(['M', 'F'])
    formato = random.choice(['3D', 'Normal'])
    precio = round(random.uniform(6, 9), 2) if formato == 'Normal' else round(random.uniform(10, 14), 2)
    
    return {
        "nombre": fake.name_male() if genero == 'M' else fake.name_female(),
        "dni": fake.unique.random_int(min=10000000, max=99999999),
        "genero": genero,
        "pelicula": random.choice(movies_list),
        "formato": formato,
        "precio": precio,
        "fecha_visita": fake.date_between(start_date='-1y', end_date='today')
    }

datos = [generar_registro() for _ in range(1000)]
df_visitors = pd.DataFrame(datos)

# Guardar en RawData y CleanData
df_visitors.to_csv("RawData/visitors_raw.csv", index=False)
df_visitors.to_csv("CleanData/visitors_clean.csv", index=False)
print("✅ Faker: 1000 registros generados.")

---

<a id='-union'></a>
## 🔹 5: Integración y Transformación Final

Unimos ambas fuentes de datos para enriquecer la tabla de visitantes con la calificación técnica de la película de IMDb.

In [ ]:
# Unión de DataFrames
df_master = pd.merge(df_visitors, df_movies, left_on='pelicula', right_on='Título', how='left')

# Limpieza post-unión (Eliminar columna duplicada)
df_master = df_master.drop(columns=['Título'])

# Guardar resultado final
df_master.to_csv('TransformData/master_data_final.csv', index=False)
print("✅ Integración completada y guardada en TransformData.")

---

<a id='-warehouse'></a>
## 🔹 6: Mini Data Warehouse (SQLite3)

Un **Data Warehouse** centraliza datos de múltiples fuentes para reportes. Usaremos SQLite por su eficiencia y portabilidad.

In [ ]:
import sqlite3

# Cargar datos en la base de datos
conn = sqlite3.connect('warehouse.db')
df_master.to_sql('ticket_sales', conn, if_exists='replace', index=False)

print("📦 Warehouse listo: Datos cargados en warehouse.db (tabla 'ticket_sales')")
conn.close()

---

<a id='-viz'></a>
## 🔹 7: Visualizaciones BI

Simulamos un entorno de Business Intelligence extrayendo datos con SQL.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect('warehouse.db')

# Consulta 1: Conteo por formato
df_formato = pd.read_sql("SELECT formato, COUNT(*) as cantidad FROM ticket_sales GROUP BY formato", conn)

# Consulta 2: Distribución de géneros
df_genero = pd.read_sql("SELECT genero, COUNT(*) as total FROM ticket_sales GROUP BY genero", conn)

conn.close()

# Gráfico
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=df_formato, x='formato', y='cantidad', ax=ax[0], palette='magma')
ax[0].set_title('Preferencias de Formato (3D vs Normal)')

sns.barplot(data=df_genero, x='genero', y='total', ax=ax[1], palette='coolwarm')
ax[1].set_title('Distribución de Visitantes por Género')

plt.show()